This notebook prepares the genomic data for imputation of the blood meQTL data from the EPIC DB database to UKB genotypes. **This notebooks is adapted from 01_genoQC.ipynb**
- get bgen variant ids (chr:pos_a1_a2) that correspond to EBB variant ids (chr:pos:a1:a2); note: alleles may be swapped so get chr:pos:a1:a2 and chr:pos:a2:a1 EBB variant ids and find the intersect wiht bgen ids.
- extract the bgen variant ids (so i dont have to update the large bgen files)
- Sample and variant QC
- update the bgen variant ids with the ebb variant ids
- Date: 30.03.2026


## Setup

In [1]:
%%bash
# pip install openpyxl
# pip install -U scikit-learn
# pip install statsmodels

In [2]:
import os
import glob
import pandas as pd
from pandas.core.common import flatten
import re
import numpy as np
import seaborn as sns
# from sklearn import datasets, linear_model, metrics
# import statsmodels.api as sm
# import statsmodels.formula.api as smf

In [28]:
%%bash
rm plink*
rm toy*
rm prettify
rm LICENSE
wget https://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20250819.zip 2>/dev/null
unzip plink_linux_x86_64_20250819.zip 2>/dev/null
./plink --version

Archive:  plink_linux_x86_64_20250819.zip
  inflating: plink                   
  inflating: LICENSE                 
  inflating: toy.ped                 
  inflating: toy.map                 
  inflating: prettify                
PLINK v1.9.0-b.7.11 64-bit (19 Aug 2025)


In [30]:
%%bash
rm intel-simplified-software-license.txt
rm plink2*
rm vcf_subset*
wget https://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_20260311.zip 2>/dev/null
unzip plink2_linux_x86_64_20260311.zip 2>/dev/null
./plink2 --version

rm: cannot remove 'plink2*': No such file or directory


Archive:  plink2_linux_x86_64_20260311.zip
  inflating: plink2                  
  inflating: vcf_subset              
  inflating: intel-simplified-software-license.txt  
PLINK v2.0.0-a.7LM 64-bit Intel (11 Mar 2026)


### load blood meQTL weights 

In [3]:
%%bash
dx download -f vasilis/data/ebb/weights/EPICDB.BLOOD.METHYL.HERIT.tar.bz2

In [4]:
%%bash
tar -xjf EPICDB.BLOOD.METHYL.HERIT.tar.bz2

### load UKB imputation data

In [5]:
%%bash
mkdir -p imp/
dx download -f -o imp/ Bulk/Imputation/'UKB imputation from genotype'/ukb22828_c*_b0_v3.mfi.txt 

## Filter samples

- no sex missmatch: p22001 vs p31
- white british ancestry: p22006
- no heterozygosiry or missingness outlier: p22027
- no \>\=10 3rd degree relatives in dataset: p22021
- no sex chromosome aneuploidy: p22019

***See 01_genoQC.ipynp. Use the QC'ed samples created there: vasilis/data/ebb/wb.eids***


## Extract weights' variants & QC variants

In [10]:
%%bash

>epicdb.variants.temp

while IFS= read -r name
do
    file=EPICDB.BLOOD.METHYL.HERIT/scores/input/$name
    if [ -f "$file" ]; then
        awk 'NR > 1 {print $1}' $file >> epicdb.variants.temp
    else
        echo "File not found: $file"
    fi
done < EPICDB.BLOOD.METHYL.HERIT/scores/scores.list

In [12]:
%%bash

## get EPICDB variant ids
cat epicdb.variants.temp | sort | uniq > epicdb.variants.1
## also get the swapped variant ids
cat epicdb.variants.1 | awk 'BEGIN{FS=":"} {print $1":"$2":"$4":"$3}' | sort | uniq > epicdb.variants.2

wc -l epicdb.variants.1
wc -l epicdb.variants.2

323249 epicdb.variants.1
323249 epicdb.variants.2


In [13]:
%%bash
mkdir -p imp/temp
mkdir -p imp/extract
for chr in {1..22}
do
    # filter INFO > 0.8 & MAF > 0.01; create mQTL variant format: chr:pos:a1:a2; sort (for joining)
    awk -v chr=$chr '$NF > 0.8 && $6 > 0.01 { print chr":"$3":"$4":"$5, $2}' imp/ukb22828_c${chr}_b0_v3.mfi.txt | sort | uniq > imp/temp/temp_c${chr}.variants
    # find epicdb variants in ukb (intersect)
    join -1 1 -2 1 epicdb.variants.1 imp/temp/temp_c${chr}.variants > imp/extract/imp_c${chr}.extract1
    join -1 1 -2 1 epicdb.variants.2 imp/temp/temp_c${chr}.variants > imp/extract/imp_c${chr}.extract2
    # keep both formats
    cat imp/extract/imp_c${chr}.extract1 imp/extract/imp_c${chr}.extract2 | awk '{print $0}' | sort | uniq > imp/extract/imp_c${chr}.bothIDs.extract
    # keep only ukb format
    awk '{print $2}' imp/extract/imp_c${chr}.bothIDs.extract | sort | uniq > imp/extract/imp_c${chr}.extract
    wc -l imp/extract/imp_c${chr}.extract
    rm imp/extract/imp_c${chr}.extract1 imp/extract/imp_c${chr}.extract2
done

28642 imp/extract/imp_c1.extract
22441 imp/extract/imp_c2.extract
19049 imp/extract/imp_c3.extract
14065 imp/extract/imp_c4.extract
17598 imp/extract/imp_c5.extract
24748 imp/extract/imp_c6.extract
17134 imp/extract/imp_c7.extract
15971 imp/extract/imp_c8.extract
9357 imp/extract/imp_c9.extract
18252 imp/extract/imp_c10.extract
17987 imp/extract/imp_c11.extract
16374 imp/extract/imp_c12.extract
8596 imp/extract/imp_c13.extract
10394 imp/extract/imp_c14.extract
9110 imp/extract/imp_c15.extract
12393 imp/extract/imp_c16.extract
13773 imp/extract/imp_c17.extract
5037 imp/extract/imp_c18.extract
10999 imp/extract/imp_c19.extract
8889 imp/extract/imp_c20.extract
4498 imp/extract/imp_c21.extract
7526 imp/extract/imp_c22.extract


In [14]:
%%bash
head -n2 imp/extract/imp_c22.extract # -> for plink 
echo ""
head -n2 imp/extract/imp_c22.bothIDs.extract # -> for matching

22:17587746_GGGC_G
22:17593972_AG_A

22:17074622:A:C rs5746664
22:17083152:G:A rs112886011


In [16]:
%%bash
n=$(cat imp/extract/imp_c*.bothIDs.extract | wc -l)
nebb=$(cat epicdb.variants.1 | wc -l)

echo "$n out of $nebb ($((100*n/nebb)) %) weights' variants in UKB imputed data (INFO > 0.8 & MAF > 0.01)"

312864 out of 323249 (96 %) weights' variants in UKB imputed data (INFO > 0.8 & MAF > 0.01)


In [ ]:
!dx upload -r imp/extract/ --dest vasilis/data/ebb/extract_blood/

In [ ]:
%%bash
dx upload mwas_01_extract.sh --dest vasilis/SAK_scripts/

In [20]:
%%bash
bgen_dir="/Bulk/Imputation/UKB imputation from genotype"
datadir="vasilis/data/ebb"
dest="vasilis/data/ebb/imp_bed_blood/"

for CHR in {1..21}; do
    dx run swiss-army-knife \
        -iin="vasilis/SAK_scripts/mwas_01_extract.sh" \
        -iin="${bgen_dir}/ukb22828_c${CHR}_b0_v3.bgen" \
        -iin="${bgen_dir}/ukb22828_c${CHR}_b0_v3.sample" \
        -iin="${datadir}/wb.eids" \
        -iin="${datadir}/extract_blood/imp_c${CHR}.extract" \
        -icmd="sh mwas_01_extract.sh ${CHR}" \
        --tag="ext_${CHR}" \
        --instance-type "mem1_ssd1_v2_x72" \
        --destination="${dest}" \
        --brief --yes --priority high
done


job-J755B8QJZB74X8v5gKpkBQyZ
job-J755B8jJZB74X8v5gKpkBVQ6
job-J755B90JZB74X8v5gKpkBVQ8
job-J755B98JZB7711Qf24Y3f5Jx
job-J755B9jJZB72VgxgFFbvF2Q7
job-J755BB0JZB74X8v5gKpkBbJB
job-J755BBQJZB72VgxgFFbvF2Q9
job-J755BBjJZB72VgxgFFbvF2QG
job-J755BF0JZB7G7BF6G9ZKxv7y
job-J755BF8JZB7P0ZfG6yFYGYfb
job-J755BFQJZB7P0ZfG6yFYGYfg
job-J755BFjJZB7P0ZfG6yFYGYfp
job-J755BG0JZB7P0ZfG6yFYGYfy
job-J755BGQJZB7G7BF6G9ZKxv8q
job-J755BGjJZB7339JPz49KqQY1
job-J755BJ0JZB74X8v5gKpkBx7k
job-J755BJ8JZB74X8v5gKpkBx7q
job-J755BJjJZB7339JPz49KqQYg
job-J755BK0JZB7711Qf24Y3f5Qx
job-J755BK8JZB7711Qf24Y3f5Qz
job-J755BKjJZB7339JPz49KqQYp


## Update variant ids 

- wait for variant extraction in the previous step to finish

####  load QC'ed UKB imputation data

In [32]:
%%bash
mkdir -p qc_imp/
dx download -f -o qc_imp/ vasilis/data/ebb/imp_bed_blood/*

#### load variant ids EBB <-> UKB tables

In [33]:
%%bash
mkdir -p extract/
dx download -f -o extract/ vasilis/data/ebb/extract_blood/* 

In [34]:
%%bash
nqc1=$(cat extract/imp_c*.bothIDs.extract | wc -l)
nqc2=$(cat qc_imp/imp_wb_qc_c*.bim | wc -l)
nebb=$(awk '{print $1}' epicdb.variants.1 | sort -u | wc -l)

echo "$nqc1 out of $nebb ($((100*nqc1/nebb)) %) weights' variants in UKB imputed data (INFO > 0.8 & MAF > 0.01)"
echo "$nqc2 out of $nebb ($((100*nqc2/nebb)) %) weights' variants in QC'ed UKB imputed data (INFO > 0.8 & MAF > 0.01 & plink QC)"

312864 out of 323249 (96 %) weights' variants in UKB imputed data (INFO > 0.8 & MAF > 0.01)
279419 out of 323249 (86 %) weights' variants in QC'ed UKB imputed data (INFO > 0.8 & MAF > 0.01 & plink QC)


In [35]:
%%bash
mkdir -p qc_imp/new_id/logs

for chr in {1..22}
do
    # remove duplicates
    awk '!seen[$2]++' extract/imp_c${chr}.bothIDs.extract > extract/imp_c${chr}.bothIDs.extract.nodup
    ./plink2 --bfile qc_imp/imp_wb_qc_c${chr} --rm-dup force-first --make-bed --out qc_imp/new_id/temp.rmdup.c${chr} >/dev/null
    # update .bim file
    ./plink2 --bfile qc_imp/new_id/temp.rmdup.c${chr} --update-name extract/imp_c${chr}.bothIDs.extract.nodup 1 2 --make-bed --out qc_imp/new_id/imp_wb_qc_newid_c${chr} >/dev/null
    # clean
    mv qc_imp/new_id/*log qc_imp/new_id/logs
    rm  qc_imp/new_id/temp.rmdup.c*
    
    echo "finished chr ${chr}"
done 

finished chr 1
finished chr 2
finished chr 3
finished chr 4
finished chr 5
finished chr 6
finished chr 7
finished chr 8
finished chr 9
finished chr 10
finished chr 11
finished chr 12
finished chr 13
finished chr 14
finished chr 15
finished chr 16
finished chr 17
finished chr 18
finished chr 19
finished chr 20
finished chr 21
finished chr 22


### Merge genotype files

In [36]:
%%bash
rm -f list_beds.txt
for chr in {1..22}; do echo "qc_imp/new_id/imp_wb_qc_newid_c${chr}" >> list_beds.txt; done

./plink \
  --merge-list list_beds.txt \
  --make-bed --out qc_imp/new_id/imp_wb_qc_newid_all


PLINK v1.9.0-b.7.11 64-bit (19 Aug 2025)           cog-genomics.org/plink/1.9/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to qc_imp/new_id/imp_wb_qc_newid_all.log.
Options in effect:
  --make-bed
  --merge-list list_beds.txt
  --out qc_imp/new_id/imp_wb_qc_newid_all

70303 MB RAM detected; reserving 35151 MB for main workspace.


to length-80+ variant IDs; consider using a different naming scheme for long
indels and the like.
position.
position.


Performing 2-pass merge (407606 people, 240836/279086 variants per pass).
Pass 1 complete.                              
Merged fileset written to qc_imp/new_id/imp_wb_qc_newid_all-merge.bed +
qc_imp/new_id/imp_wb_qc_newid_all-merge.bim +
qc_imp/new_id/imp_wb_qc_newid_all-merge.fam .
279086 variants loaded from .bim file.
407606 people (187273 males, 220333 females) loaded from .fam.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 407606 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989 done.
Total genotyping rate is 0.988242.
279086 variants and 407606 people pass filters and QC.
Note: No phenotypes present.
--make-bed to qc_imp/new_id/imp_wb_qc_newid_all.bed +
qc_imp/new_id/imp_wb_qc_newid_all.bim + qc_imp/new_id/imp_wb_qc_newid_all.fam
... 1011121314

In [ ]:
# upload merged data
!dx upload --brief qc_imp/new_id/imp_wb_qc_newid_all* --dest vasilis/data/ebb/qc_imp/new_id_blood/